# QRGuard Structural v3 (LATEST)

One `structural-2026.03-r01` ResNet-18 artifact is evaluated on both Gallery and Camera. Capture conditions are nuisance/quality slices, not malicious labels. The notebook never promotes or deploys the candidate automatically.

## Phase 0 - Choose exactly one run mode

In [ ]:
RUN_MODE = 'fresh'  # fresh | resume | evaluate_only | report_only
RUN_ID = 'r01'        # choose a new ID for another fresh experiment
VERSION = 'structural-2026.03-r01'
VALID_MODES = {'fresh', 'resume', 'evaluate_only', 'report_only'}
if RUN_MODE not in VALID_MODES:
    raise ValueError(f'RUN_MODE must be one of {sorted(VALID_MODES)}')
print('Mode:', RUN_MODE, '| Version:', VERSION, '| Run ID:', RUN_ID)


## Phase 1 - Reproducible source bundle and Drive

In [ ]:
# Mount Drive and unpack the exact source bundle.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

BUNDLE_ZIP = Path('/content/drive/MyDrive/QRGuard_ML_Colab.zip')
WORK = Path('/content/qrguard_ml')
if not BUNDLE_ZIP.is_file():
    raise FileNotFoundError(
        f'Upload QRGuard_ML_Colab.zip to {BUNDLE_ZIP} before Run all.'
    )
print('Bundle SHA-256:', hashlib.sha256(BUNDLE_ZIP.read_bytes()).hexdigest().upper())
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as archive:
    archive.extractall(WORK)
REPO = WORK / 'QRGuard_ML_Colab' / 'QRGuard'
assert (REPO / 'ml_training/requirements.txt').is_file(), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'backend'))
print('Training source:', REPO)


In [ ]:
DRIVE_ML = Path('/content/drive/MyDrive/QRGuard_ML')
DRIVE_RUN = DRIVE_ML / 'runs' / VERSION / RUN_ID
CHECKPOINT_DIR = DRIVE_RUN / 'checkpoints'
OUTPUT_DRIVE = DRIVE_RUN / 'outputs'
CACHE_DRIVE = DRIVE_ML / 'cache' / VERSION
DATA_DRIVE = Path('/content/drive/MyDrive/QRGuard_ML_Data/structural')
PERF = REPO / 'ml_training/structural/performance' / VERSION
ARTIFACTS = REPO / 'ml_training/structural/runs' / VERSION / 'artifacts'
PROCESSED_ROOT = REPO / 'ml_training/datasets/structural/processed'
os.environ['QRGUARD_STRUCTURAL_VERSION'] = VERSION
for directory in (DRIVE_RUN, CHECKPOINT_DIR, OUTPUT_DRIVE, CACHE_DRIVE):
    directory.mkdir(parents=True, exist_ok=True)
print('Persistent run:', DRIVE_RUN)


## Phase 2 - Install only when inference or training is needed

In [ ]:
if RUN_MODE == 'report_only':
    print('report_only: dependency installation and GPU checks skipped')
else:
    # Install QRGuard dependencies without replacing Colab's matched
    # CUDA-enabled torch/torchvision wheels. Re-resolving only one side of that pair
    # can make torchvision fail during import even though `import torch` still works.
    def torch_runtime_smoke_test(stage):
        try:
            import torch, torchvision
            from torchvision import models, transforms
            probe = models.resnet18(weights=None)
            assert probe.fc.in_features == 512 and transforms.ToTensor is not None
        except Exception as exc:
            raise RuntimeError(
                f'Colab torch/torchvision is inconsistent {stage}: {exc}\n'
                'Choose Runtime > Disconnect and delete runtime, reconnect with a T4 GPU, '
                'then run this updated notebook from Phase 0.'
            ) from exc
        print(
            f'PyTorch runtime {stage}: torch={torch.__version__}, '
            f'torchvision={torchvision.__version__}, CUDA={torch.cuda.is_available()}'
        )
    
    torch_runtime_smoke_test('before dependency install')
    requirements = REPO / 'ml_training/requirements.txt'
    colab_requirements = Path('/tmp/qrguard_colab_requirements.txt')
    protected = {'torch', 'torchvision'}
    lines = []
    for line in requirements.read_text(encoding='utf-8').splitlines():
        package = line.split(';', 1)[0].split('[', 1)[0]
        package = package.split('=', 1)[0].split('<', 1)[0].split('>', 1)[0].strip().lower()
        if package not in protected:
            lines.append(line)
    colab_requirements.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        '-r', str(colab_requirements),
    ])
    torch_runtime_smoke_test('after dependency install')
    
    def run_module(module, *arguments, check=True):
        # Colab can suppress output inherited by subprocess.run. Merge and stream
        # both channels explicitly so the real child traceback is never replaced by
        # an unhelpful outer CalledProcessError.
        command = [sys.executable, '-u', '-m', module, *map(str, arguments)]
        print('>', ' '.join(command), flush=True)
        environment = os.environ.copy()
        environment['PYTHONUNBUFFERED'] = '1'
        process = subprocess.Popen(
            command,
            cwd=REPO,
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        output = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
            output.append(line)
        returncode = process.wait()
        result = subprocess.CompletedProcess(
            command, returncode, stdout=''.join(output), stderr=None
        )
        if check and returncode:
            raise RuntimeError(
                f'{module} failed with return code {returncode}; '
                'the complete child output is printed immediately above.'
            )
        return result
    
    run_module('ml_training.scripts.audit_environment')
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a T4 GPU for training/evaluation.')
    print('GPU:', torch.cuda.get_device_name(0))


## Phase 3 - Restore report, or prepare a version-locked dataset

Prepared public data is cached in Drive. The combined manifest is rebuilt only when the v3 capture manifest or config changes.

In [ ]:
if RUN_MODE == 'report_only':
    saved_performance = OUTPUT_DRIVE / 'performance'
    if saved_performance.is_dir():
        shutil.copytree(saved_performance, PERF, dirs_exist_ok=True)
        print('Restored saved Drive performance; no model training/evaluation ran.')
    elif (PERF / 'metrics.json').is_file():
        print(
            'No saved Drive report yet; displaying the bundled 2026-08-30 '
            'local CPU reference. This is not a Colab run or deployment evidence.'
        )
    else:
        raise FileNotFoundError(f'No saved or bundled report at {saved_performance}')
else:
    cached_processed = CACHE_DRIVE / 'processed'
    if cached_processed.is_dir():
        shutil.copytree(cached_processed, PROCESSED_ROOT, dirs_exist_ok=True)
        print('Restored prepared Structural data from Drive cache')

    downloads = REPO / 'ml_training/datasets/structural/downloads'
    archives = {
        DATA_DRIVE / 'QR-DN1.0.zip': downloads / 'qrdn/QR-DN1.0.zip',
        DATA_DRIVE / 'qr_codes_in_surfaces.zip': downloads / 'qr_surfaces/qr_codes_in_surfaces.zip',
    }
    public_ready = all((PROCESSED_ROOT / name / 'manifest.csv').is_file()
                       for name in ('qrdn', 'qr_surfaces'))
    if not public_ready:
        for source, destination in archives.items():
            if not source.is_file():
                raise FileNotFoundError(f'Missing official archive: {source}')
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
        run_module('ml_training.scripts.verify_datasets')
        for archive, destination in (
            (archives[DATA_DRIVE / 'QR-DN1.0.zip'], REPO / 'ml_training/datasets/structural/raw/qrdn'),
            (archives[DATA_DRIVE / 'qr_codes_in_surfaces.zip'], REPO / 'ml_training/datasets/structural/raw/qr_surfaces'),
        ):
            destination.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archive) as package:
                package.extractall(destination)
        run_module('ml_training.structural.src.prepare_qrdn')
        run_module('ml_training.structural.src.prepare_qr_surfaces')

    drive_captures = DATA_DRIVE / 'runtime_captures'
    local_captures = REPO / 'data/runtime_captures'
    local_captures.mkdir(parents=True, exist_ok=True)
    if drive_captures.is_dir():
        shutil.copytree(drive_captures, local_captures, dirs_exist_ok=True)
    capture_audit = run_module(
        'ml_training.structural.src.prepare_structural_v3_captures',
        local_captures, '--strict', check=False,
    )
    print('Real paired capture gate:',
          'READY' if capture_audit.returncode == 0 else 'NOT READY - candidate only')

    capture_manifest = local_captures / 'manifest_v3.csv'
    config_path = REPO / 'ml_training/configs' / f'{VERSION}.json'
    cache_contract_path = CACHE_DRIVE / 'cache_contract.json'
    expected_contract = {
        'version': VERSION,
        'config_sha256': hashlib.sha256(config_path.read_bytes()).hexdigest(),
        'capture_manifest_sha256': hashlib.sha256(capture_manifest.read_bytes()).hexdigest(),
    }
    recorded_contract = (
        json.loads(cache_contract_path.read_text())
        if cache_contract_path.is_file() else None
    )
    combined_manifest = PROCESSED_ROOT / VERSION / 'manifest.csv'
    if not combined_manifest.is_file() or recorded_contract != expected_contract:
        run_module(
            'ml_training.structural.src.train_local',
            '--mode', RUN_MODE,
            '--prepare-only', '--rebuild-data',
        )
        shutil.copytree(PROCESSED_ROOT, cached_processed, dirs_exist_ok=True)
        cache_contract_path.write_text(json.dumps(expected_contract, indent=2))
        print('Prepared dataset and refreshed Drive cache')
    else:
        print('Dataset cache identity matched; expensive preparation skipped')


## Phase 4 - Dataset composition and leakage audit

In [ ]:
if RUN_MODE != 'report_only':
    import pandas as pd
    manifest = pd.read_csv(PROCESSED_ROOT / VERSION / 'manifest.csv')
    display(pd.crosstab(manifest.split, [manifest.label, manifest.source]))
    groups = {name: set(part.group_id) for name, part in manifest.groupby('split')}
    overlaps = {
        f'{left}/{right}': len(groups[left] & groups[right])
        for index, left in enumerate(groups)
        for right in list(groups)[index + 1:]
    }
    assert not any(overlaps.values()), overlaps
    print('Rows:', len(manifest), '| groups:', manifest.group_id.nunique())
    if 'quality_condition' in manifest:
        display(pd.crosstab(manifest.quality_condition, manifest.label))


## Phase 5 - Train/resume/evaluate, then save outputs

Every completed epoch is checkpointed to Drive. A gate failure still produces a valid candidate report and never updates the app.

In [ ]:
if RUN_MODE != 'report_only':
    execution = run_module(
        'ml_training.structural.src.train_local',
        '--mode', RUN_MODE,
        '--checkpoint-dir', CHECKPOINT_DIR,
        check=False,
    )
    if PERF.is_dir():
        shutil.copytree(PERF, OUTPUT_DRIVE / 'performance', dirs_exist_ok=True)
    if ARTIFACTS.is_dir():
        shutil.copytree(ARTIFACTS, OUTPUT_DRIVE / 'artifacts', dirs_exist_ok=True)
    print('Execution return code:', execution.returncode)
    if not (PERF / 'metrics.json').is_file():
        raise RuntimeError('Execution stopped before a complete performance report.')


## Phase 6 - Display reusable performance evidence

In [ ]:
from IPython.display import Image as DisplayImage, Markdown, display
import pandas as pd

metrics_path = PERF / 'metrics.json'
if not metrics_path.is_file():
    raise FileNotFoundError(metrics_path)
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
if metrics.get('version') != VERSION:
    raise RuntimeError(f"Wrong report version: {metrics.get('version')}")
current_config = REPO / 'ml_training/configs' / f'{VERSION}.json'
current_config_hash = hashlib.sha256(current_config.read_bytes()).hexdigest()
if metrics.get('run_identity', {}).get('config_sha256') != current_config_hash:
    raise RuntimeError('Saved report does not match this Structural config.')
state_path = CHECKPOINT_DIR / 'run_state.json'
if state_path.is_file():
    saved_state = json.loads(state_path.read_text(encoding='utf-8'))
    if saved_state.get('identity') != metrics.get('run_identity'):
        raise RuntimeError('Saved report and checkpoint run identities disagree.')
display(Markdown((PERF / 'STRUCTURAL_PERFORMANCE.md').read_text(encoding='utf-8')))
display(pd.DataFrame([
    {'Gate': 'Research', 'Passed': metrics['research_gates_passed'],
     'Failures': '; '.join(metrics['research_gate_failures']) or 'none'},
    {'Gate': 'Deployment', 'Passed': metrics['deployment_gates_passed'],
     'Failures': '; '.join(metrics['deployment_gate_failures']) or 'none'},
]))
for name in ('training_curves.png', 'confusion_matrix.png', 'roc_pr_curves.png',
             'calibration_curve.png', 'qrdn_clean_distribution.png'):
    path = PERF / name
    if path.is_file():
        display(Markdown(f'### {name}'))
        display(DisplayImage(filename=str(path)))
for name in ('metrics.csv', 'dataset_composition.csv', 'per_source_results.csv',
             'per_device_results.csv', 'per_quality_condition_results.csv',
             'quality_abstention_results.csv', 'gallery_camera_consistency.csv',
             'exported_gallery_camera_consistency.csv',
             'exported_runtime_predictions.csv', 'misclassified_samples.csv',
             'training_history.csv'):
    path = PERF / name
    if path.is_file() and path.stat().st_size:
        display(Markdown(f'### {name}'))
        display(pd.read_csv(path).head(50))
print('Persistent output:', OUTPUT_DRIVE)


## Phase 7 - Validate report completeness (no promotion)

In [ ]:
validation = run_module(
    'ml_training.scripts.validate_performance_bundle',
    '--branch', 'structural', '--structural-version', VERSION,
    check=False,
) if RUN_MODE != 'report_only' else None
if validation is not None:
    print('Report validation return code:', validation.returncode)
print('Done. This notebook did not push, deploy, or replace runtime models.')
